# Run the matched-budget GST seed sweep

For each seed this runs adaptive FPR first, uses its accounted revealed-shot cost as the target, and calibrates the two fixed-shot comparisons toward that same budget.

In [1]:
from pathlib import Path
import subprocess
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / "seed_sweep_experiments",
    Path.cwd() / "GST_POUNDERS" / "seed_sweep_experiments",
    Path("/workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments"),
]
EXPERIMENT_DIR = next(
    (p.resolve() for p in candidates if (p / "run_matched_budget_sweep.py").exists()),
    None,
)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Could not locate seed_sweep_experiments.")

CONFIG_PATH = EXPERIMENT_DIR / "experiment_config.json"
RESULTS_DIR = EXPERIMENT_DIR / "matched_results"
print("Experiment directory:", EXPERIMENT_DIR)
print("Python:", sys.executable)

Experiment directory: /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments
Python: /usr/local/bin/python


In [2]:
RUN_SWEEP = True
SEED_SPEC = "101:102"          # Expand after one seed succeeds.
FORCE_RERUN = False
# rho_gated_geometric , lazy_delta_inverse_square

In [3]:
command = [
    sys.executable, "-u", str(EXPERIMENT_DIR / "run_matched_budget_sweep.py"),
    "--config", str(CONFIG_PATH),
    "--results-dir", str(RESULTS_DIR),
    "--seeds", SEED_SPEC,
]
if FORCE_RERUN:
    command.append("--force")

print("Command:", " ".join(command))
if not RUN_SWEEP:
    print("Dry run only. Set RUN_SWEEP=True to execute.")
else:
    process = subprocess.Popen(
        command,
        cwd=EXPERIMENT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Matched sweep exited with code {return_code}")
    print("Matched-budget sweep complete.")

Command: /usr/local/bin/python -u /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/run_matched_budget_sweep.py --config /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/experiment_config.json --results-dir /workspace/IBCDFO/GST_POUNDERS/seed_sweep_experiments/matched_results --seeds 101:102
RUN seed=101 method=fixed_fpr
[POUDERS] Beginning gradient-based optimization.
[POUDERS] Initial residual/Jacobian evaluation took 61.47 seconds.
/usr/local/lib/python3.11/site-packages/pygsti/forwardsims/mapforwardsim.py:732: UserWarning: Generating dense process matrix representations of circuits or gates 
can be inefficient and should be avoided for the purposes of forward 
simulation/calculation of circuit outcome probability distributions 
when using the MapForwardSimulator.
  _warnings.warn('Generating dense process matrix representations of circuits or gates \n'
[POUDERS] FPR reduction selected 4252/55832 residuals; union=4252/55832; active=4252/55832.
[POUDERS] Initial point evalua

KeyboardInterrupt: 

In [ ]:
import pandas as pd

summary_path = RESULTS_DIR / "matched_budget_summary.csv"
if summary_path.exists():
    matched_summary_df = pd.read_csv(summary_path)
    display(
        matched_summary_df[ 
            [
                "data_seed",
                "method",
                "target_revealed_shots",
                "actual_revealed_shots",
                "relative_budget_difference",
                "weighted_least_squares_objective",
                "mean_gate_entanglement_infidelity_to_truth",
                "mean_spam_vector_l2_error_to_truth",
            ]
        ].sort_values(["data_seed", "method"]).reset_index(drop=True)
    )
else:
    print("No matched-budget summary exists yet.")

Open analyze_detailed_accuracy.ipynb and set RESULTS_DIR to matched_results to plot the matched-budget runs.